# Fiche 7 — Servir un modèle avec FastAPI, et le tester

| | |
|---|---|
| 🎯 **Objectif** | Écrire une API avec une route POST, valider les entrées avec Pydantic et la tester avec `TestClient`, sans lancer de serveur |
| ⏱️ **Durée** | 50 min |
| 🧰 **Prérequis** | Fonctions, dictionnaires ; notion de requête HTTP |
| 🔗 **Dans le projet** | Étape 3 (API `/predict` et `/recommend`) |

> **L'idée clé.** Une API reçoit du JSON, le **valide**, appelle le modèle et renvoie du JSON. La validation est le travail principal : une entrée absurde doit être refusée avec un message clair (code 422), pas transformée en prédiction absurde.

In [1]:
import warnings

warnings.filterwarnings("ignore", message=".*httpx.*")  # avertissement interne de la bibliothèque de test

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

# Un « modèle » minuscule : rendement habituel × un effet de la température (pour l'exemple)
HABITUEL = {"blé": 2.5, "maïs": 2.9, "riz": 3.3}


def predire(culture, temperature):
    return HABITUEL[culture] * (1 + 0.01 * (temperature - 15))

## 1. Une API en quelques lignes

In [2]:
app = FastAPI(title="API de démonstration")


class Demande(BaseModel):
    culture: str
    temperature: float = Field(..., ge=-15, le=45, description="°C, moyenne annuelle")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(d: Demande):
    if d.culture not in HABITUEL:
        raise HTTPException(status_code=422, detail=f"Culture inconnue : {d.culture}")
    return {"culture": d.culture, "rendement_t_ha": round(predire(d.culture, d.temperature), 2)}

`Demande` décrit ce qu'on attend : une culture (texte) et une température entre −15 et 45 °C. FastAPI vérifie
chaque requête contre ce modèle **avant** d'appeler `predict`, et génère la documentation interactive (`/docs`).

## 2. Tester sans serveur avec `TestClient`

In [3]:
client = TestClient(app)
print(client.get("/health").json())
r = client.post("/predict", json={"culture": "blé", "temperature": 20})
print(r.status_code, r.json())

{'status': 'ok'}
200 {'culture': 'blé', 'rendement_t_ha': 2.62}


## ⚠️ Piège : sans validation, une entrée absurde devient une prédiction absurde

In [4]:
print("température 300 °C →", client.post("/predict", json={"culture": "blé", "temperature": 300}).status_code)
print("température 'chaud' →", client.post("/predict", json={"culture": "blé", "temperature": "chaud"}).status_code)
print("culture inconnue    →", client.post("/predict", json={"culture": "banane", "temperature": 20}).json())
print("sans validation, predire('blé', 300) =", round(predire("blé", 300), 1), "t/ha (!)")

température 300 °C → 422
température 'chaud' → 422
culture inconnue    → {'detail': 'Culture inconnue : banane'}
sans validation, predire('blé', 300) = 9.6 t/ha (!)


Avec Pydantic, les trois requêtes sont refusées (code **422**, « entité non traitable ») avec un message. La
fonction brute, elle, renvoie sans broncher un rendement de blé de plus de 9 t/ha à 300 °C.

### ✍️ À toi de jouer

Ajoute une route `POST /recommend` qui reçoit seulement `temperature` et renvoie la liste des cultures triées par rendement prédit décroissant. Écris deux tests : la liste contient 3 cultures, et elle est bien triée.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
class Contexte(BaseModel):
    temperature: float = Field(..., ge=-15, le=45)

@app.post("/recommend")
def recommend(c: Contexte):
    classement = sorted(HABITUEL, key=lambda k: predire(k, c.temperature), reverse=True)
    return [{"culture": k, "rendement_t_ha": round(predire(k, c.temperature), 2)} for k in classement]

r = TestClient(app).post("/recommend", json={"temperature": 20}).json()
assert len(r) == 3
assert [x["rendement_t_ha"] for x in r] == sorted([x["rendement_t_ha"] for x in r], reverse=True)
print(r)
```

**Pourquoi ?** une route = une fonction ; les tests vérifient le **contrat** (forme et ordre de la réponse), pas seulement « ça répond ». Dans le projet, le classement se fait sur le rendement **relatif** (fiche 9), pas sur les tonnes brutes, et `main.py` reste mince : la logique est dans le package `agrireco`, testée à part.
</details>

## ✅ À retenir

1. Une API = des routes (GET pour lire, POST pour envoyer des données à traiter).
2. Pydantic décrit les entrées attendues ; FastAPI refuse le reste avec un code 422.
3. `TestClient` teste l'API sans lancer de serveur : idéal pour la CI.
4. Le fichier de l'API reste mince : il valide, appelle le package, renvoie du JSON.
5. Une erreur métier (culture inconnue) renvoie un message clair, pas une erreur 500.